# Data Quality Assessment

Data Quality Assessment is the process of checking a dataset for missing information, duplicate records, invalid values, and inconsistencies before using it for analysis or machine learning.

# Missing Values

## Number of missing values
Definition: The number of missing values is the total count of empty or unavailable entries in a dataset or column. In Pandas, missing values are commonly represented by NaN, None, or NaT, depending on the data type.

### Real-world example:
An employee database has 12 employees whose salary information has not been entered.

### AI/ML use case: 
Identify missing features before training a model so that incomplete data can be handled appropriately.

In [2]:
%mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 4.592 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
+ python-tzdata  2026.5   pyh5ded981_0         conda-forge
- pip            26.2.1   pyh145f28c_0         conda-forge


In [3]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["Ravi", "Priya", "Arun", "Meena"],
    "Age": [25, np.nan, 30, np.nan],
    "Salary": [30000, 40000, np.nan, 50000]
})

missing_count = df.isnull().sum()

print(missing_count)
print("Total missing values:", df.isnull().sum().sum())

Name      0
Age       2
Salary    1
dtype: int64
Total missing values: 3


### Explanation
- df.isnull() marks missing cells as True and available cells as False.
- .sum() counts missing cells in each column.
- .sum().sum() adds those column counts to get the total number of missing cells in the entire DataFrame.
- Here, Age has 2 missing values and Salary has 1, giving 3 missing values overall.

## Percentage of missing values
Definition: The percentage of missing values measures how much of a column or dataset is incomplete. It is calculated by dividing the number of missing entries by the total number of entries and multiplying by 100.

### Real-world example: 
If 20 out of 100 customer records have no email address, the email column has 20% missing values.

### AI/ML use case: 
Decide whether a feature should be imputed, investigated, or potentially removed because too much information is missing.

In [5]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Age": [25, np.nan, 30, np.nan],
    "Salary": [30000, 40000, np.nan, 50000],
    "City": ["Kadapa", "Chennai", "Bengaluru", "Hyderabad"]
})

missing_percentage = df.isnull().mean() * 100

print(missing_percentage.round(2))

Age       50.0
Salary    25.0
City       0.0
dtype: float64


### Explanation
- df.isnull() identifies the missing entries.
- .mean() calculates the proportion of missing entries in each column because True is treated as 1 and False as 0.
- Multiplying by 100 converts the proportion into a percentage.
- Age is 50% missing, Salary is 25% missing, and City has no missing values.

## Missing values by column
Definition: Missing values by column means examining each column separately to determine how many entries are missing. This helps identify which features have data-quality problems.

### Real-world example: 
A hospital dataset may have complete patient names but many missing blood pressure or weight readings.

### AI/ML use case: 
Prioritize features that need cleaning and decide which columns require imputation or further investigation.

In [7]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Name": ["Ravi", "Priya", "Arun", "Meena"],
    "Age": [25, np.nan, 30, np.nan],
    "Salary": [np.nan, 40000, np.nan, 50000],
    "City": ["Kadapa", "Chennai", None, "Hyderabad"]
})

missing_by_column = df.isnull().sum()

print("Missing values by column:")
print(missing_by_column)

print("\nColumns containing missing values:")
print(missing_by_column[missing_by_column > 0])

Missing values by column:
Name      0
Age       2
Salary    2
City      1
dtype: int64

Columns containing missing values:
Age       2
Salary    2
City      1
dtype: int64


### Explanation
- isnull().sum() counts missing entries separately for each column.
- missing_by_column > 0 selects only columns that contain missing values.
- This makes it easier to identify which features need attention instead of inspecting every row manually.

## Missing value patterns
Definition: Missing value patterns describe how missing entries are distributed across rows and columns, including whether multiple fields are missing together. They help distinguish random gaps from systematic problems in data collection.

### Real-world example: 
Customers who register through a mobile app may have missing postal addresses more frequently than customers who register in a store.

### AI/ML use case: 
Investigate whether missingness is associated with particular groups or features, which can affect model bias and imputation choices.

In [8]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Age": [25, np.nan, 30, np.nan],
    "Salary": [30000, np.nan, 40000, 50000],
    "City": ["Kadapa", "Chennai", np.nan, np.nan]
})

# Show True for missing cells
print("Missing-value pattern:")
print(df.isnull())

# Count rows with each missingness pattern
pattern_counts = (
    df.isnull()
      .astype(int)
      .value_counts()
)

print("\nPattern counts (1 = missing, 0 = present):")
print(pattern_counts)

Missing-value pattern:
     Age  Salary   City
0  False   False  False
1   True    True  False
2  False   False   True
3   True   False   True

Pattern counts (1 = missing, 0 = present):
Age  Salary  City
0    0       0       1
1    1       0       1
0    0       1       1
1    0       1       1
Name: count, dtype: int64


### Explanation
- df.isnull() creates a Boolean table showing which cells are missing.
- .astype(int) converts False to 0 and True to 1.
- .value_counts() counts the distinct row patterns.
- For example, the pattern Age=1, Salary=1, City=0 means both age and salary are missing while city is present.
- This helps detect recurring gaps and investigate whether they are linked to a particular process or group.

# Duplicate Records

## Identify duplicates
Definition: Identifying duplicates means finding rows or records that repeat information already present in a dataset. Pandas uses duplicated() to mark repeated rows or repeated values in selected columns.

### Real-world example: 
An online store accidentally imports the same customer's order twice.

### AI/ML use case: 
Detect duplicate training examples that could distort class distributions or cause data leakage between training and test sets.

In [9]:
import pandas as pd

df = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 102],
    "Name": ["Ravi", "Priya", "Arun", "Priya"],
    "City": ["Kadapa", "Chennai", "Bengaluru", "Chennai"]
})

print(df.duplicated())
print("\nDuplicate customer IDs:")
print(df.duplicated(subset=["Customer_ID"]))

0    False
1    False
2    False
3     True
dtype: bool

Duplicate customer IDs:
0    False
1    False
2    False
3     True
dtype: bool


### Explanation
- df.duplicated() checks whether an entire row has appeared earlier, considering all columns by default.
- The first occurrence is marked False; a later matching occurrence is marked True.
- subset=["Customer_ID"] checks only the customer ID, even if other columns differ.
- By default, duplicated() keeps the first occurrence and marks subsequent duplicates.

## Count duplicates
Definition: Counting duplicates means calculating how many records Pandas identifies as repeated under a specified matching rule. This can be done for complete rows or selected columns.

### Real-world example: 
A business discovers that 350 customer records repeat an earlier customer ID.

### AI/ML use case: 
Measure duplicate-data problems before cleaning a dataset and evaluate how much repeated data may affect model training.

In [10]:
import pandas as pd

df = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 102, 104, 101],
    "Name": ["Ravi", "Priya", "Arun", "Priya", "Meena", "Ravi"]
})

total_duplicates = df.duplicated().sum()
duplicate_ids = df.duplicated(subset=["Customer_ID"]).sum()

print("Completely duplicated rows:", total_duplicates)
print("Duplicate customer IDs:", duplicate_ids)

Completely duplicated rows: 2
Duplicate customer IDs: 2


### Explanation
- df.duplicated().sum() counts rows that repeat an earlier row across all columns.
- df.duplicated(subset=["Customer_ID"]).sum() counts repeated customer IDs after their first occurrence.
- Both counts are 2 in this example, but they represent different matching rules.
- The number of duplicate IDs does not necessarily equal the number of duplicate complete rows in other datasets.

## Identify completely duplicated rows
Definition: Completely duplicated rows are records where every column has the same value as an earlier record. They can be displayed using the Boolean mask produced by duplicated().

### Real-world example: 
An employee import contains two identical rows for the same person, department, and salary.

### AI/ML use case: 
Remove accidental repeated observations when each row is intended to represent a unique event or record.

In [11]:
import pandas as pd

df = pd.DataFrame({
    "Employee_ID": [1, 2, 3, 2],
    "Name": ["Ravi", "Priya", "Arun", "Priya"],
    "Salary": [30000, 40000, 50000, 40000]
})

duplicates = df[df.duplicated(keep=False)]

print(duplicates)

   Employee_ID   Name  Salary
1            2  Priya   40000
3            2  Priya   40000


### Explanation
- duplicated(keep=False) marks every occurrence of a repeated row, including the first one.
- df[...] filters the DataFrame to show only those rows.
- Both copies of Priya's record are displayed because all three columns match.
- To remove repeated complete rows, you can use df.drop_duplicates() after confirming that removing them is appropriate.

## Identify potential duplicate records
Definition: Potential duplicates are records that may refer to the same real-world entity even though their values are not exactly identical. They are detected by checking selected identifiers or comparing similar names, contact details, addresses, and other fields.

### Real-world example: 
The same customer appears as “Ravi Kumar” and “Ravi K.” with the same phone number.

### AI/ML use case:
Prepare clean customer datasets for entity resolution, deduplication, and customer-level prediction tasks.

In [12]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Ravi Kumar", "Priya Reddy", "Ravi K.", "Arun"],
    "Phone": ["9876543210", "9123456780",
              "9876543210", "9988776655"],
    "City": ["Kadapa", "Chennai", "Kadapa", "Bengaluru"]
})

# Flag repeated phone numbers
df["Potential_Duplicate"] = df.duplicated(
    subset=["Phone"],
    keep=False
)

print(df[df["Potential_Duplicate"]])

         Name       Phone    City  Potential_Duplicate
0  Ravi Kumar  9876543210  Kadapa                 True
2     Ravi K.  9876543210  Kadapa                 True


### Explanation
- subset=["Phone"] checks whether a phone number appears more than once.
- keep=False flags every record sharing that phone number.
- Both Ravi records are flagged even though their names differ.
- These are potential duplicates, not automatically confirmed duplicates. Shared family phone numbers, recycled numbers, or data-entry mistakes can create false matches, so business rules or manual review may be needed.

# Invalid Values

## Negative values where inappropriate
Definition: Negative values are invalid when a fields real-world meaning does not allow them, such as a person's age or a product's physical quantity in stock. Whether a negative number is wrong depends on the business meaning of the column.

### Real-world example: 
A store's inventory report shows -12 units in stock even though negative stock is not permitted by its rules.

### AI/ML use case: 
Prevent impossible numerical values from distorting statistics, feature engineering, and model predictions.

In [13]:
import pandas as pd

df = pd.DataFrame({
    "Product": ["Laptop", "Phone", "Tablet", "Mouse"],
    "Stock": [10, -5, 20, -2]
})

invalid_stock = df[df["Stock"] < 0]

print(invalid_stock)

  Product  Stock
1   Phone     -5
3   Mouse     -2


### Explanation
- df["Stock"] < 0 creates a Boolean condition for negative stock values.
- df[...] returns only the rows that satisfy the condition.
- Two products are flagged for investigation.
- Negative values should not always be removed: fields such as account balances, temperature changes, and profit or loss may legitimately be negative.

## Impossible values
Definition: Impossible values are values that violate the logical or physical limits of a field. They may be detected using domain rules, such as age ranges, valid quantities, or measurements with known limits.

### Real-world example: 
A customer database records a person's age as 250 years.

### AI/ML use case: 
Detect anomalous or erroneous measurements before training a model so it learns from plausible observations.

In [14]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Ravi", "Priya", "Arun", "Meena"],
    "Age": [25, 250, -3, 40]
})

# Example rule: age must be between 0 and 120
invalid_age = df[
    ~df["Age"].between(0, 120)
]

print(invalid_age)

    Name  Age
1  Priya  250
2   Arun   -3


### Explanation
- .between(0, 120) identifies ages within the chosen inclusive range.
- ~ reverses the condition to select ages outside the range.
- Priya's age of 250 and Arun's age of -3 are flagged.
- The range is an example validation rule, not a universal rule for every dataset. Invalid records should be corrected, marked, or investigated rather than silently changed.

## Incorrect categories
Definition: Incorrect categories are values that do not follow the allowed labels or naming conventions for a categorical field. They often result from spelling mistakes, inconsistent capitalization, abbreviations, or unexpected labels.

### Real-world example: 
A customer database contains "Hydrabad" instead of "Hyderabad" and "bengaluru" instead of "Bengaluru".

### AI/ML use case: 
Prevent inconsistent category labels from creating unintended separate categories during encoding and model training.

In [15]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Ravi", "Priya", "Arun", "Meena"],
    "City": ["Kadapa", "Hydrabad", "Bengaluru", "UnknownCity"]
})

valid_cities = ["Kadapa", "Hyderabad", "Bengaluru", "Chennai"]

invalid_categories = df[
    ~df["City"].isin(valid_cities)
]

print(invalid_categories)

    Name         City
1  Priya     Hydrabad
3  Meena  UnknownCity


### Explanation
- valid_cities contains the permitted city names for this example.
- .isin(valid_cities) checks whether each city belongs to that list.
- ~ selects values that are not in the permitted set.
- "Hydrabad" and "UnknownCity" are flagged.
- A value can be valid but absent from the list if the list is incomplete. Confirm the allowed categories before changing or removing any values.

## Invalid dates
Definition: Invalid dates are date values that cannot be parsed correctly or do not follow the expected date format or business rules. Examples include impossible calendar dates and dates that fall outside an allowed time range.

### Real-world example: 
An order record contains 2026-02-30, which is not a valid calendar date.

### AI/ML use case: 
Ensure that time-based features, such as customer tenure, order delays, and time-series forecasting inputs, are calculated correctly.

In [16]:
import pandas as pd

df = pd.DataFrame({
    "Order_ID": [101, 102, 103, 104],
    "Order_Date": [
        "2026-01-15",
        "2026-02-30",
        "15-03-2026",
        "not_a_date"
    ]
})

df["Parsed_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

invalid_dates = df[df["Parsed_Date"].isna()]

print(invalid_dates[["Order_ID", "Order_Date"]])

   Order_ID  Order_Date
1       102  2026-02-30
3       104  not_a_date


### Explanation
- pd.to_datetime() attempts to convert the text values into valid datetime values.
- format="mixed" allows Pandas to infer the format separately for each value.
- dayfirst=True helps interpret dates such as 15-03-2026 as day-month-year.
- errors="coerce" converts values that cannot be parsed into NaT, Pandas' missing datetime value.
- .isna() identifies the records that failed conversion.
- A date may be syntactically valid but still violate a business rule, such as an order date occurring in the future. Such checks require additional validation.

## Incorrect data types
Definition: Incorrect data types occur when a column is stored in a format that does not match its intended meaning. For example, an age may be stored as text, or a date may be stored as a string instead of a datetime value.

### Real-world example: 
An employee's age is stored as "25" instead of a numeric value, making numerical calculations more difficult.

### AI/ML use case: 
Ensure that numerical calculations, date operations, and feature encoding work correctly before building a model.

In [17]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Ravi", "Priya", "Arun"],
    "Age": ["25", "30", "unknown"],
    "Salary": ["30000", "45000", "50000"]
})

print("Original data types:")
print(df.dtypes)

# Convert columns to numeric
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")

print("\nUpdated data types:")
print(df.dtypes)

print("\nConverted DataFrame:")
print(df)

Original data types:
Name      str
Age       str
Salary    str
dtype: object

Updated data types:
Name          str
Age       float64
Salary      int64
dtype: object

Converted DataFrame:
    Name   Age  Salary
0   Ravi  25.0   30000
1  Priya  30.0   45000
2   Arun   NaN   50000


### Explanation
- Initially, Age and Salary are stored as text (object).
- pd.to_numeric() converts values that can be interpreted as numbers.
- errors="coerce" converts unrecognized values such as "unknown" into NaN.
- The resulting columns can now be used for numerical calculations.
- The invalid age should still be investigated or handled; converting the type does not automatically fix the underlying data.

## Unexpected values
Definition: Unexpected values are values that are technically valid but violate the expected range, format, or business rules for a particular field. They are commonly identified using validation rules and domain-specific checks.

### Real-world example: 
An online store records a product discount of 150%, even though discounts are allowed only from 0% to 100%.

### AI/ML use case: 
Detect unusual inputs that could distort predictions or indicate data-entry errors, fraud, or changes in the data-generation process.

In [18]:
import pandas as pd

df = pd.DataFrame({
    "Product": ["Laptop", "Phone", "Tablet", "Mouse"],
    "Discount": [10, 25, 150, -5]
})

# Business rule: discount must be between 0 and 100
unexpected = df[
    ~df["Discount"].between(0, 100)
]

print(unexpected)

  Product  Discount
2  Tablet       150
3   Mouse        -5


### Explanation
- The business rule permits discounts from 0% to 100%, inclusive.
- .between(0, 100) checks whether each value is within the permitted range.
- ~ selects records that violate the rule.
- The 150% and -5% discounts are flagged for investigation.
- Unlike a general impossible-value check, unexpected-value checks often depend on the specific business context. An unusual value is not automatically wrong.

## Empty strings
Definition: An empty string is a text value containing no characters (""), which is different from a standard Pandas missing value such as NaN. Empty strings can make a field appear populated even when it contains no useful information.

### Real-world example: 
A customer form submits an empty email field, and the database stores it as "" rather than a missing value.

### AI/ML use case: 
Ensure blank text fields are recognized as missing before performing imputation, text processing, or categorical encoding.

In [19]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Ravi", "", "Priya", "   "],
    "Email": ["ravi@example.com", "", "priya@example.com", " "]
})

# Replace whitespace-only strings with missing values
df = df.replace(r"^\s*$", pd.NA, regex=True)

print("DataFrame after cleaning:")
print(df)

print("\nMissing values:")
print(df.isnull().sum())

DataFrame after cleaning:
    Name              Email
0   Ravi   ravi@example.com
1    NaN                NaN
2  Priya  priya@example.com
3    NaN                NaN

Missing values:
Name     2
Email    2
dtype: int64


### Explanation
- r"^\s*$" is a regular expression that matches a string containing only whitespace, including an empty string.
- replace(..., pd.NA, regex=True) converts those values into Pandas missing values.
- isnull().sum() counts the missing entries in each column.
- Both empty strings and strings containing only spaces are now recognized as missing.
- This makes subsequent missing-value analysis more reliable.

## Null-like values
Definition: Null-like values are text entries such as "NULL", "null", "N/A", "None", or "unknown" that represent missing or unavailable information but are stored as ordinary strings. Pandas does not automatically treat every such string as a missing value in every context.

### Real-world example: 
A CSV file stores a missing customer phone number as the text "N/A" rather than as a true missing value.

### AI/ML use case: 
Standardize missing-value representations so that missing data is not incorrectly treated as a meaningful category or text feature.

In [20]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Ravi", "Priya", "Arun", "Meena"],
    "Phone": ["9876543210", "N/A", "NULL", "9123456780"],
    "City": ["Kadapa", "None", "Chennai", "unknown"]
})

null_like_values = [
    "N/A", "NULL", "None", "unknown"
]

# Replace matching text values with proper missing values
df = df.replace(null_like_values, pd.NA)

print(df)

print("\nMissing values:")
print(df.isnull().sum())

    Name       Phone     City
0   Ravi  9876543210   Kadapa
1  Priya         NaN      NaN
2   Arun         NaN  Chennai
3  Meena  9123456780      NaN

Missing values:
Name     0
Phone    2
City     2
dtype: int64


### Explanation
- null_like_values defines the text values that your dataset uses to represent missing information.
- df.replace(null_like_values, pd.NA) replaces matching values with a standard missing-value marker.
- isnull().sum() then counts these values as missing.
- The list should be chosen carefully because words like "None" or "unknown" may be meaningful in some datasets rather than representing missing information.